# Splunk Data Analysis — ForgeRock / Ping Identity logs

Run SPL searches through the Splunk Python SDK, pull results into pandas, and do root-cause analysis on AM/IDM/DS logs: error spikes, failing journeys, slow queries, recon failures.

> Credentials from environment variables (`SPLUNK_HOST`, `SPLUNK_PORT`, `SPLUNK_USERNAME`, `SPLUNK_PASSWORD`).

## 1. Imports

In [ ]:
import os

import pandas as pd
import splunklib.client as client
import splunklib.results as results
from rich.console import Console

console = Console()
pd.set_option('display.max_columns', 50)

## 2. Connect

In [ ]:
service = client.connect(
    host=os.environ.get('SPLUNK_HOST', 'splunk.example.com'),
    port=int(os.environ.get('SPLUNK_PORT', '8089')),
    username=os.environ['SPLUNK_USERNAME'],
    password=os.environ['SPLUNK_PASSWORD'],
)
print('connected to', service.host, '| apps:', len(service.apps))

## 3. Run a search
Example: ForgeRock AM authentication failures over the last 24h. Swap the SPL for whatever incident you're chasing.

In [ ]:
EARLIEST = '-24h'
LATEST = 'now'
SPL = ('search index=fr_index sourcetype=am-authentication result=FAILED '
       '| table _time, user, realm, authType, failureReason, nodeId')

job = service.jobs.create(SPL, earliest_time=EARLIEST, latest_time=LATEST)
print('search job started:', job.sid)
job.refresh()
while not job.is_done():
    job.refresh()  # big jobs: add time.sleep + a progress bar here
print('done —', job['resultCount'], 'results')

## 4. Results into a DataFrame

In [ ]:
rows = [dict(r) for r in results.ResultsReader(job.results(count=0)) if isinstance(r, dict)]
df = pd.DataFrame(rows)
if '_time' in df.columns:
    df['_time'] = pd.to_datetime(df['_time'])
    df = df.sort_values('_time').reset_index(drop=True)
print(df.shape)
df.head()

## 5. Root-cause analysis
First looks: what fails most, who it hits, and when it spikes.

In [ ]:
# top failure reasons — the 'why'
if 'failureReason' in df.columns:
    print(df['failureReason'].value_counts().head(10))

In [ ]:
# most-affected users / identities — the 'who'
for col in ('user', 'authType', 'realm'):
    if col in df.columns:
        print(f'--- by {col} ---')
        print(df[col].value_counts().head(10))

In [ ]:
# failures per hour — the 'when'; spikes point at deployments or upstream outages
if '_time' in df.columns and len(df):
    hourly = df.assign(hour=df['_time'].dt.floor('h')).groupby('hour').size()
    print('peak:', hourly.idxmax(), '->', hourly.max(), 'failures')
    hourly.tail(24)

## 6. Reusable patterns
- **Recon failures:** `sourcetype=idm-recon result=FAILURE | stats count by mapping, failureReason`
- **Slow IDM queries:** filter `durationMs > 1000`, group by endpoint — missing indexes show up fast
- **OTP storms:** join AM journey executions with the Twilio burst list from `twilio_otp_analysis.ipynb`

Save the DataFrame (`df.to_csv(...)`) and the same analysis code reruns against any search — that repeatability is the whole point of doing Splunk work in Jupyter instead of the search bar.